In [1]:
import os
from dotenv import load_dotenv

from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from langsmith import Client
from langchain.tools import tool
import requests

In [18]:
# Load environment variables from .env file
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [19]:
@tool
def get_weather_data(city: str) -> str:
    """Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temparature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%\n"
    )
 

In [5]:
search_tool = TavilySearch(max_results=2)
llm = ChatOpenAI(model="gpt-4o-mini")


In [6]:
result = search_tool.invoke("Give me the latest news on AI")
result

{'query': 'Give me the latest news on AI',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://guides.library.georgetown.edu/ai/news',
   'title': 'Guides: Artificial Intelligence (Generative) Resources: Latest News on AI',
   'content': "U.A.E.-Based AI and Data Startup Expands Into Silicon Valley; CNTXT AI, which raised $60 million in June, has built Arabic-first AI models and is training robots This link opens in a new window_Rosenbush, Steven Wall Street Journal (Online) 19 Oct 2026Formats: Citation,Full text,Find Full Text @ GU,Find Full Text @ GU_ Oct 19, 2026 [...] Nvidia-Backed Startup doubleAI Launches Model to Pare Cost of Compute; The company, co-founded by Mobileye CEO Amnon Shashua, says it slashed one early customer's compute costs by 67%This link opens in a new window_Lin, Belle Wall Street Journal (Online) 19 Oct 2026Formats: Citation,Full text,Find Full Text @ GU,Find Full Text @ GU_ Oct 19, 2026 [...] DHS Orders DeepSeek Technol

In [9]:
# LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=OPENAI_API_KEY
)

In [10]:
response = llm.invoke("Tell me about a joke of AI")
response

AIMessage(content='Sure! Here’s a classic AI-themed joke for you:\n\nWhy did the robot go on a diet?\n\nBecause it had too many bytes!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 14, 'total_tokens': 42, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_242228cf5a', 'id': 'chatcmpl-Ec7tPNk4dzCUYu6xbJKBE8GulbYGA', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1688e-2ed5-760e-9d13-682134e89b9d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 28, 'total_tokens': 42, 'input_token_details': {'audio': 

In [11]:
# prompt from LangSmith Prompt Hub
client = Client()
prompt = client.pull_prompt("hwchase17/react", dangerously_pull_public_prompt=True)

In [12]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [7]:
tools = [search_tool, get_weather_data]

In [13]:
# create agent
from  langchain_classic.agents import create_react_agent, AgentExecutor
agent = create_react_agent(
    llm, 
    tools=tools,
    prompt=prompt)

In [14]:
# executor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True # whether to print out intermediate steps or not(logs)
)

In [21]:
# run
response = agent_executor.invoke({
    "input": (
        "Find the capital of France"
        "and then find its current weather."
    )
})



> Entering new AgentExecutor chain...
I need to find the capital of France first, which I know is Paris. After that, I will check the current weather in Paris.  
Action: get_weather_data  
Action Input: "Paris"  City: Paris
Temparature: 12°C
Weather: Light rain shower
Humidity: 82%
I now know the final answer.  
Final Answer: The capital of France is Paris, and the current weather there is 12°C with light rain showers and 82% humidity.

> Finished chain.
